# 02 - Cleaning: which laps are representative?

**Question of this notebook:** which laps can be used to measure the pace of
a driver, and which must be left out?

A lap is **not clean** if at least one of these six rules applies:

| Rule | Why | Column |
|---|---|---|
| pit lap | the in-lap and the out-lap are slowed by the pit lane | `pin` / `pout` not missing |
| lap 1 | standing start, cars bunched up | `lap == 1` |
| not green | safety car, virtual safety car or yellow flag | `status != "1"` |
| deleted | cancelled by the stewards | `del` |
| inaccurate | start and end of the lap not synced | `iacc` False |
| no time | last lap of a driver who stopped | `time` missing |

The rows are **kept**: the function adds a true/false column `is_clean`.
The strategy question needs every lap (all stints, pit laps included), and
pace and degradation filter with `is_clean`.

In [1]:
import os
import sys

# The notebooks live in notebooks/, the package in the project root.
sys.path.append(os.path.abspath(os.path.join("..")))

# Reload boxbox automatically when a .py file changes.
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

In [3]:
from boxbox.cleaning import mark_clean_laps
from boxbox.data import get_laps, prepare_laps

In [4]:
prepared = prepare_laps(get_laps(2022, "bahrain"))
print(len(prepared), "laps")

1125 laps


## 1. How many laps does each rule affect?

This is the exploration that led to the function: one condition per rule
(a copy of the rules in `mark_clean_laps`, which is the reference).

In [5]:
rules = {
    "pit lap": prepared["pin"].notna() | prepared["pout"].notna(),
    "lap 1": prepared["lap"] == 1,
    "not green": prepared["status"] != "1",
    "deleted": prepared["del"],
    "inaccurate": ~prepared["iacc"],
    "no time": prepared["time"].isna(),
}

per_rule = pd.Series({name: bad.sum() for name, bad in rules.items()})
per_rule

pit lap       117
lap 1          20
not green     167
deleted         4
inaccurate    232
no time         2
dtype: int64

In [6]:
any_bad = pd.concat(rules, axis=1).any(axis=1)
flags = per_rule.sum()

print("flags in total:", flags)
print("laps removed:  ", any_bad.sum(), "(the rules overlap)")
print("clean laps:    ", (~any_bad).sum(), "of", len(prepared))

flags in total: 542
laps removed:   271 (the rules overlap)
clean laps:     854 of 1125


### Does `iacc` remove anything on its own?

`iacc == False` on 232 laps (21% of the race) looks like a lot. The question
is how many of them are **not** already removed by another rule.

In [7]:
others = pd.concat(
    {name: bad for name, bad in rules.items() if name != "inaccurate"}, axis=1
).any(axis=1)

only_iacc = rules["inaccurate"] & ~others
print(only_iacc.sum(), "laps removed only by iacc")

0 laps removed only by iacc


**Result on Bahrain 2022: 0.** Every inaccurate lap is already removed by
another rule, so `iacc` adds nothing here. It is kept because it costs one
line and may matter on another race; this is stated for this race only.

## 2. The function

`mark_clean_laps` returns a copy of the table with the column `is_clean`.

In [8]:
laps = mark_clean_laps(prepared)
print(laps["is_clean"].sum(), "clean of", len(laps))  # expect 854 of 1125
print(f"{laps['is_clean'].mean():.1%} of the laps are clean")

854 clean of 1125
75.9% of the laps are clean


In [9]:
# The function and the exploration above must agree.
print((laps["is_clean"] == ~any_bad).all())

True


In [10]:
# The input table was not modified.
print("is_clean" in prepared.columns)

False


## 3. Do the status codes match the lap times?

The codes come from the documentation, and a multi-digit code lists what
happened during the lap **in order** (`"41"` = safety car, then clear). To
check that they are read correctly, the lap times of each status are
compared. Pit laps and lap 1 are left out here, because they are slow for
another reason.

If the codes are right, the laps with a safety car (`4`, `41`, `64`, `164`)
must be much slower than the green laps (`1`).

In [11]:
no_pit_no_start = laps["pin"].isna() & laps["pout"].isna() & (laps["lap"] != 1)

by_status = (
    laps[no_pit_no_start]
    .groupby("status")["time"]
    .agg(["count", "median"])
    .sort_values("median")
)
by_status

,count,median
status,,
12,28,99.0385
1,858,99.3275
2,7,100.3950
16,3,103.5620
4,61,119.0930
164,7,124.7850
64,3,128.3310
41,19,135.0420


**What the first exploration showed** (all laps, pit laps included, so the
numbers were a little higher): green laps (`1`) about 99.7 s; safety-car
laps (`4`, `41`, `64`, `164`) between 123 s and 135 s; a virtual safety car
(`16`) about 105 s; yellow only (`2`, `12`, `21`) between 99 and 103 s.

So the codes are read correctly: the safety car makes a lap 20 to 36 seconds
slower. Laps with only a yellow flag are close to green laps. The strict rule
(`status == "1"`) removes them anyway; a looser rule is a possible
robustness check later.

## 4. A second race: Monaco 2022

Everything was built on Bahrain. A second race shows whether a rule behaves
badly elsewhere. Things to look at: a red flag (status `5`), wet-weather
tyres (`INTERMEDIATE`, `WET`), and a much smaller share of clean laps.

In [12]:
monaco = mark_clean_laps(prepare_laps(get_laps(2022, "monaco")))
print(monaco.shape, monaco["drv"].nunique(), "drivers")
print(f"{monaco['is_clean'].mean():.1%} of the laps are clean")

(1179, 15) 20 drivers
70.7% of the laps are clean


In [13]:
print(monaco["compound"].value_counts())
print()
print(monaco["status"].value_counts())

compound
HARD            393
MEDIUM          346
WET             283
INTERMEDIATE    115
SOFT              7
Name: count, dtype: int64

status
1       910
12      149
4        36
14       18
451      18
64       17
126      16
21        8
124       2
41        2
264       1
26        1
1264      1
Name: count, dtype: int64


In [14]:
# Clean laps per driver: a driver with very few is left out of the pace table.
monaco[monaco["is_clean"]].groupby("drv").size().sort_values()

drv
MAG    12
MSC    14
ALB    29
TSU    43
NOR    44
GAS    44
HAM    44
RUS    45
VER    45
LEC    46
VET    46
PER    46
RIC    46
ZHO    46
LAT    47
OCO    47
SAI    47
STR    47
ALO    47
BOT    48
dtype: int64

**Monaco 2022 (1179 laps, 70.7% clean).** Compounds: HARD 393, MEDIUM 346, WET 283, INTERMEDIATE 115, SOFT 7, and 35 laps with no compound at all (`value_counts` hides them). Status: all of `4` (safety car), `5` (red flag, in `451`) and `6` (VSC) appear, plus yellow-flag laps (`12`, 149 laps). The strict rule removes them all.
The 70% rule (limit 34 clean laps, from a maximum of 48) excludes MAG (12), MSC (14) and ALB (29), which it did not do in Bahrain.
**Conclusion:** the cleaning rules work, but the race mixes wet and dry laps, so a median over the whole race says nothing about pace. Bahrain 2022 (dry) stays the main race. The 35 laps without a compound are looked at below.

In [15]:
print(monaco["compound"].isna().sum())
monaco[monaco["compound"].isna()][["drv", "lap", "stint", "status", "time"]].head(10)

35


,drv,lap,stint,status,time
349,STR,30,5,1,103.833
350,STR,31,5,1,84.515
351,STR,32,5,1,97.937
352,STR,33,5,1,84.605
353,STR,34,5,1,82.704
354,STR,35,5,1,82.363
355,STR,36,5,1,82.264
356,STR,37,5,1,81.741
357,STR,38,5,1,81.665
358,STR,39,5,1,80.557


The 35 laps without a compound all belong to one driver (STR, stint 5, from lap 30). Their status and times look normal (the times fall from 84 s to 80.5 s as the track dries), so the source has no compound for that stint. Cleaning is not affected (no rule uses the compound); strategy and degradation would have to leave these laps out and say so.